In [1]:
import os
import json
import torch
import networkx as nx
import pandas as pd
import numpy as np

from tqdm import tqdm

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch_geometric.nn import GCNConv
from torch.nn import Linear

import torch.nn.functional as F

In [2]:
dataset_df = pd.read_csv(

    "../data/processed/full_composition_dataset.csv"
)

dataset_df.head()

,image_name,brightness_mean,brightness_std,edge_density,balance_score,subject_count,spatial_spread
0,10003.jpg,126.447526,31.343642,0.023579,0.0,0,NaN
1,10007.jpg,99.889099,78.729215,0.009401,NaN,2,148.604385
2,10008.jpg,150.302874,53.160128,0.051828,0.0,0,NaN
3,1001.jpg,48.357921,57.923257,0.056185,NaN,3,119.610544
4,10010.jpg,100.815889,62.652103,0.144118,NaN,17,255.902569


In [13]:
graph_path = (
    "C:/Users/bhanu/Downloads/StructCompose/data/processed/graphs/"
    + dataset_df.iloc[5]["image_name"]
    + ".json"
)

with open(graph_path, "r") as f:

    graph_data = json.load(f)

G = nx.node_link_graph(graph_data)

print(G)

Graph with 2 nodes and 1 edges


In [14]:
def graph_to_data(G, target):

    node_features = []

    for node, attrs in G.nodes(data=True):

        feature_vector = [

            attrs["center_x"],
            attrs["center_y"],
            attrs["area"],
            attrs["confidence"]

        ]

        node_features.append(feature_vector)

    x = torch.tensor(
        node_features,
        dtype=torch.float
    )

    edge_index = []

    for u, v in G.edges():

        edge_index.append([u, v])
        edge_index.append([v, u])

    edge_index = torch.tensor(
        edge_index,
        dtype=torch.long
    ).t().contiguous()

    y = torch.tensor(
        [target],
        dtype=torch.float
    )

    data = Data(
        x=x,
        edge_index=edge_index,
        y=y
    )

    return data

In [15]:
graph_dataset = []

In [16]:
for idx, row in tqdm(dataset_df.iterrows()):

    image_name = row["image_name"]

    graph_path = (
        "../data/processed/graphs/"
        + image_name
        + ".json"
    )

    if not os.path.exists(graph_path):
        continue

    try:

        with open(graph_path, "r") as f:

            graph_data = json.load(f)

        G = nx.node_link_graph(graph_data)

        if len(G.nodes()) == 0:
            continue

        target = row["edge_density"]

        data = graph_to_data(
            G,
            target
        )

        graph_dataset.append(data)

    except Exception as e:

        print(image_name, e)

9497it [01:48, 87.68it/s] 


In [17]:
print(len(graph_dataset))

print(graph_dataset[0])

7461
Data(x=[2, 4], edge_index=[2, 2], y=[1])


In [18]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(

    graph_dataset,

    test_size=0.2,

    random_state=42
)

In [19]:
train_loader = DataLoader(
    train_data,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=16
)

In [27]:
from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

In [28]:
class SCGNet(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(4, 32)

        self.conv2 = GCNConv(32, 64)

        self.fc1 = Linear(64, 32)

        self.fc2 = Linear(32, 1)

    def forward(self, data):

        x = data.x

        edge_index = data.edge_index

        batch = data.batch

        x = self.conv1(x, edge_index)

        x = F.relu(x)

        x = self.conv2(x, edge_index)

        x = F.relu(x)

        # IMPORTANT FIX
        x = global_mean_pool(x, batch)

        x = self.fc1(x)

        x = F.relu(x)

        x = self.fc2(x)

        return x

In [29]:
device = torch.device(

    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = SCGNet().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = torch.nn.MSELoss()

In [30]:
epochs = 20

In [31]:
for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        output = model(batch)

        loss = criterion(
            output.squeeze(),
            batch.y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} | Loss: {avg_loss:.4f}"
    )

Epoch 1 | Loss: 128975.4267
Epoch 2 | Loss: 1239.7499
Epoch 3 | Loss: 506.1745
Epoch 4 | Loss: 37328.6247
Epoch 5 | Loss: 24.9502
Epoch 6 | Loss: 4.8356
Epoch 7 | Loss: 181.9397
Epoch 8 | Loss: 5017.3951
Epoch 9 | Loss: 810.3196
Epoch 10 | Loss: 10996.8837
Epoch 11 | Loss: 15590.3252
Epoch 12 | Loss: 1.1709
Epoch 13 | Loss: 0.8208
Epoch 14 | Loss: 2.3823
Epoch 15 | Loss: 11520.4211
Epoch 16 | Loss: 1518.1546
Epoch 17 | Loss: 0.3055
Epoch 18 | Loss: 1.2902
Epoch 19 | Loss: 0.2701
Epoch 20 | Loss: 0.6143


In [32]:
model.eval()

predictions = []
targets = []

with torch.no_grad():

    for batch in test_loader:

        batch = batch.to(device)

        output = model(batch)

        predictions.extend(
            output.cpu().numpy()
        )

        targets.extend(
            batch.y.cpu().numpy()
        )

In [33]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(
    targets,
    predictions
)

print("Test MSE:", mse)

Test MSE: 2.6566693418559435
